In [14]:
import pandas as pd
import sqlalchemy as sal
import pyodbc

engine = sal.create_engine(r"mssql+pyodbc://AKSHAYKUMAR\SQLEXPRESS01/master?"r"driver=ODBC+DRIVER+18+FOR+SQL+SERVER&"r"TrustServerCertificate=yes")
conn = engine.connect()

In [15]:
def extract():
    df_products = pd.read_csv('products.csv')
    df_products_db = pd.read_sql_query("select * from product_dim where end_date = '9999-12-31' ", conn)
    return df_products, df_products_db

def transform(df_products, df_products_db):
    df_merged = pd.merge(df_products, df_products_db, how='inner', on = 'product_id')
    return df_merged

def inserts(df_products):
    df_products.to_sql('product_dim', con=conn, index=False, if_exists='append')
    conn.commit()

def updates(product_keys):
    query = sal.text("update product_dim set end_date = cast(getdate()-1 as date) where product_key in (" +product_keys+ ")")
    p = conn.execute(query)
    conn.commit()

In [16]:
df_products, df_products_db = extract()

In [17]:
df_merged = transform(df_products, df_products_db)

In [18]:
update_rows = df_merged['product_key']

In [19]:
keys = update_rows.to_list()
product_keys = ','.join([str(key) for key in keys])

In [20]:
if product_keys != '':
    updates(product_keys)

In [21]:
df_products['start_date'] = pd.to_datetime('now').strftime('%Y-%m-%d')
df_products['end_date'] = '9999-12-31'

In [22]:
inserts(df_products)